# Jinja and Macros

## Overview
dbt models are compiled through **Jinja2** before being sent to the database. This enables dynamic SQL that adapts to the environment, project variables, and reusable helper logic.

**Jinja syntax in dbt:**

| Syntax | Purpose |
|---|---|
| `{{ expression }}` | Output a value (variable, function call, macro result) |
| `{% statement %}` | Control flow: `if`, `for`, `set`, `macro` |
| `{# comment #}` | Comment (not compiled into SQL) |

**Key dbt Jinja functions:**

| Function | Purpose |
|---|---|
| `ref('model')` | Reference another dbt model |
| `source('src', 'table')` | Reference a raw source |
| `var('name')` | Access a project variable |
| `env_var('NAME')` | Read an OS environment variable |
| `target.name` / `target.type` | Current target metadata |
| `run_started_at` | Timestamp when this dbt run began |
| `this` | Current model's fully-qualified table name (used in incremental models) |

**Macros** are Jinja functions defined in `.sql` files in the `macros/` directory. They compile to SQL at runtime and accept arguments with optional defaults.

---

In [ ]:
examples = [
    ("Variable interpolation",
     "SELECT * FROM orders WHERE status = value_from_var",
     "Use var() to inject a project variable at compile time"),
    ("if/else",
     "if target.name == prod: longer lookback\nelse: shorter lookback for dev",
     "Different filter windows in dev vs prod to reduce dev query cost"),
    ("set", "{% set schema = analytics %}\nSELECT * FROM schema.customers",
     "Local Jinja variable assignment"),
]
print("=== Jinja in dbt SQL files ===")
for title, code, note in examples:
    print(f"  --- {title} ---")
    print(code)
    print(f"  -> {note}")
    print()

for_loop_sql = (
    "SELECT order_id,\n"
    "{% for method in var('payment_methods') %}\n"
    "  SUM(CASE WHEN payment_method = '{{ method }}' THEN amount ELSE 0 END) AS {{ method }}_amount\n"
    "  {% if not loop.last %},{% endif %}\n"
    "{% endfor %}\n"
    "FROM payments GROUP BY order_id"
)
print("For loop example (one SUM column per payment method):")
print(for_loop_sql)

if_else_sql = (
    "SELECT * FROM raw_page_views\n"
    "{% if target.name == 'prod' %}\n"
    "  WHERE created_at >= CURRENT_DATE - INTERVAL '7 days'\n"
    "{% else %}\n"
    "  WHERE created_at >= CURRENT_DATE - INTERVAL '1 day'\n"
    "{% endif %}"
)
print()
print("if/else example:")
print(if_else_sql)

---
## Writing and using custom macros

In [ ]:
macro_cents = (
    "-- macros/cents_to_dollars.sql\n"
    "{% macro cents_to_dollars(column_name, decimal_places=2) %}\n"
    "    ROUND({{ column_name }} / 100.0, {{ decimal_places }})\n"
    "{% endmacro %}"
)
macro_divide = (
    "-- macros/safe_divide.sql\n"
    "{% macro safe_divide(numerator, denominator) %}\n"
    "    CASE WHEN {{ denominator }} = 0 OR {{ denominator }} IS NULL\n"
    "         THEN NULL\n"
    "         ELSE {{ numerator }}::NUMERIC / {{ denominator }}\n"
    "    END\n"
    "{% endmacro %}"
)
usage = (
    "-- Model SQL using macros:\n"
    "SELECT order_id,\n"
    "    {{ cents_to_dollars('total_amount_cents') }}  AS total_amount_usd,\n"
    "    {{ safe_divide('refund_cents', 'total_cents') }} AS refund_rate\n"
    "FROM {{ ref('int_order_payments_joined') }}\n\n"
    "-- Compiles to:\n"
    "SELECT order_id,\n"
    "    ROUND(total_amount_cents / 100.0, 2)  AS total_amount_usd,\n"
    "    CASE WHEN total_cents = 0 OR total_cents IS NULL THEN NULL\n"
    "         ELSE refund_cents::NUMERIC / total_cents END AS refund_rate\n"
    "FROM analytics.staging.int_order_payments_joined"
)
print("=== Custom macros ===")
print(macro_cents)
print(macro_divide)
print()
print("=== Macro usage in a model ===")
print(usage)

---
## var(), env_var(), and runtime overrides

In [ ]:
print("=== var() and env_var() ===")
vars_yaml = (
    "# dbt_project.yml\n"
    "vars:\n"
    "  payment_methods: [credit_card, coupon, bank_transfer, gift_card]\n"
    "  start_date: '2018-01-01'\n"
    "  is_test_run: false"
)
print(vars_yaml)
print()
patterns = [
    ("var('start_date')",                  "Injects '2018-01-01' from project vars"),
    ("var('start_date', '2020-01-01')",    "With default fallback if var not set"),
    ("if var('is_test_run'): LIMIT 100",    "Conditional LIMIT in dev"),
    ("env_var('DBT_USER')",                 "Reads from OS env; errors if not set"),
    ("env_var('DBT_SCHEMA', 'analytics')", "Env var with default fallback"),
]
for expr, note in patterns:
    print(f"  {expr:<48s}  # {note}")
print()
print("Runtime override: dbt run --vars '{start_date: 2023-01-01}'")

---
## Common Pitfalls

**1. Using Jinja `{{ }}` in YAML without quoting**
In YAML, curly braces must be wrapped in quotes: `values: "{{ var('payment_methods') }}"`. Without quotes, YAML parsers interpret them as mapping syntax and throw a parse error.

**2. Forgetting that Jinja runs at compile time, not at query runtime**
The `if target.name` block is evaluated when dbt compiles the SQL, not when the database executes it. You cannot branch on runtime data values using Jinja; it only controls what SQL is generated.

**3. Macros with no type validation producing silent wrong output**
`cents_to_dollars("amount")` compiles to `ROUND(amount / 100.0, 2)` even if `amount` is already in dollars. Macros have no type safety. Document expected input types in docstrings.

**4. Hardcoding `target.name` instead of `target.type` for engine-specific SQL**
`if target.name == "prod"` breaks for teams that name production differently. Use `target.type == "postgres"` for database-specific logic and a consistent naming convention for environment branches.

**5. Storing secrets in var() instead of env_var()**
`var("db_password")` is stored in `dbt_project.yml` which lives in the git repo. Always use `env_var("DBT_PASSWORD")` for credentials and inject them via your CI/CD secrets manager.

**6. Large for loops generating hundreds of CASE WHEN columns**
A for loop over a 500-item variable list generates 500 CASE WHEN clauses. This overwhelms some SQL parsers and produces unmaintainable compiled SQL. Use JSON aggregation or Python-side pivoting for unbounded column lists.


---
*sql_methods_library - Samantha McGarrigle*